In [ ]:
import glob
import os
from pathlib import Path
from random import choice, choices

import autoroot
import hydra
import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch
from omegaconf import OmegaConf

from src.data.lf.lf_datamodule import LF
from src.modules.lit_module import LitModule
from src.modules.models.sequential import SequentialModel
from src.utils import initialize_resolvers
from src.utils.tools import get_rho_lstsq

initialize_resolvers()

In [ ]:
def get_lastest_ckpt(name):
    path = "./logs/train"
    path = Path(path)
    ckpt = max((path / name / "checkpoints").glob("*.ckpt"), key=os.path.getctime)
    return ckpt


def data_illustration(inputs, outputs, index=0, yhat=None):

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].plot(inputs[index, :, 0].numpy(), label="x")
    ax[0].plot(outputs[index, :, 0].numpy(), label="y")
    if yhat is not None:
        ax[0].plot(yhat[index, :, 0].numpy(), label="yhat")
    ax[0].legend()

    ax[1].plot(get_rho_lstsq(inputs, outputs, end_point=False)[:, 0], label="rho")
    if yhat is not None:
        ax[1].plot(get_rho_lstsq(inputs, yhat, end_point=False)[:, 0], label="rho_hat")
    ax[1].legend()

In [ ]:
dt = 0.05
data = lambda noise: LF(
    data_dir="",
    N=200,
    T=128,
    D=1,
    rho_name="exponential",
    rho_param={"alpha": 1},
    dt=dt,
    input_generator="randn",
    processing=noise,
    manual_seed=0,
).dataset_construction()

In [ ]:
index = 2

### RNN, No noise

 train/loss_epoch: 6.12e-06 val/loss: 5.81e-06

In [ ]:
name = "rnn-no_noise"
data1 = data(None)
model = LitModule.load_from_checkpoint(get_lastest_ckpt(name))
yhat = model(data1[:][0].to("cuda:0")).cpu().detach()
data_illustration(data1[:][0], data1[:][1], index=index, yhat=yhat)

### RNN, Have noise

train/loss_epoch: 2.335e-1 val/loss: 2.389e-1

In [ ]:
name = "rnn-noise"
data1 = data(["noise"])
model = LitModule.load_from_checkpoint(get_lastest_ckpt(name))
yhat = model(data1[:][0].to("cuda:0")).cpu().detach()
data_illustration(data1[:][0], data1[:][1], index=index, yhat=yhat)

### gatedRNN, Have noise

train/loss_epoch: 5.50e-02 val/loss: 5.47e-02

In [ ]:
name = "gatedRNN-noise"
data1 = data(["noise"])
model = LitModule.load_from_checkpoint(get_lastest_ckpt(name))
yhat = model(data1[:][0].to("cuda:0")).cpu().detach()
data_illustration(data1[:][0], data1[:][1], index=index, yhat=yhat)

$$h_{t+1}=g1(x_t)h_t+g2(x_t)x_t$$

$$sigmoid(A x_t+b)$$

### Generalize to other type of Noise

### gatedRNN, No noise

train/loss_epoch: 1.38e-05 val/loss: 1.17e-05

In [ ]:
index = 0
name = "gatedRNN-no_noise"
data1 = data(None)
model = LitModule.load_from_checkpoint(get_lastest_ckpt(name))
yhat = model(data1[:][0].to("cuda:0")).cpu().detach()
data_illustration(data1[:][0], data1[:][1], index=index, yhat=yhat)

In [ ]:
index = 3
name = "gatedRNN-noise"
data1 = data(["noise", "noise"])
model = LitModule.load_from_checkpoint(get_lastest_ckpt(name))
yhat = model(data1[:][0].to("cuda:0")).cpu().detach()
data_illustration(data1[:][0], data1[:][1], index=index, yhat=yhat)

### Ablation 

$$h_{t+1} = h_t + g(x_t) x_t  $$
$$h_{t+1} = (1-g_t)h_t + g(x_t) x_t  $$

In [ ]:
name = "gatedRNN-noise"
data1 = data(["noise"])
model = LitModule.load_from_checkpoint(get_lastest_ckpt(name))
yhat = model(data1[:][0].to("cuda:0"), return_g=True).cpu().detach()
data_illustration(data1[:][0], data1[:][1], index=index, yhat=yhat)